# Exploration: Time Stamps

Parsing the timestamps and understanding them

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt 
from pathlib import Path
from anomaly_detection.etl.load import load_timestamps, load_timestamps_df, load_records

plt.style.use('ggplot')

In [ ]:
# Path to the data
project_folder = Path.cwd().parent

evtx_path = project_folder / "data/raw/93_applog.evtx"

evtx_path

In [ ]:
output_path = project_folder / "data/time_exploration/application"

output_path

In [ ]:
timestamps = load_timestamps(evtx_path)

len(timestamps)

In [ ]:
timestamps_df = load_timestamps_df(evtx_path)

timestamps_df.head(10)

In [ ]:
# Export timestamps_df to csv and describe it
timestamps_df.to_csv(project_folder / "data/processed/timestamps.csv", index=False)

timestamps_df.describe(include='all')

In [ ]:
# Count events per minute and events per hour for plotting
events_per_minute = (
    timestamps_df
    .groupby("minute")
    .size()
    .rename("events")
    .reset_index()
)

events_per_hour = (
    timestamps_df
    .groupby("hour")
    .size()
    .rename("events")
    .reset_index()
)

events_per_hour

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(
    events_per_minute["minute"],
    events_per_minute["events"],
    color="red"
)

plt.title("Number of Events per Minute")
plt.xlabel("Time")
plt.ylabel("Events")

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(output_path / "events_per_minute.png")

plt.show()

In [ ]:
events_by_minute_of_hour = (
    timestamps_df
    .groupby("minute_of_hour")
    .size()
    .reindex(range(60), fill_value=0)
    .rename("events")
    .reset_index()
)

events_by_hour_of_day = (
    timestamps_df
    .groupby("hour_of_day")
    .size()
    .reindex(range(24), fill_value=0)
    .rename("events")
    .reset_index()
)

events_by_hour_of_day

In [ ]:
events_by_minute_of_hour_dict = events_by_minute_of_hour.to_dict(orient='records')

events_by_minute_of_hour_dict = [{str(record['minute_of_hour']).zfill(2): record['events']} for record in events_by_minute_of_hour_dict]

with open(output_path / "events_per_minutes_of_hour.txt", "w", encoding="utf-8") as file:
    file.write(json.dumps(
        events_by_minute_of_hour_dict,
        indent=4
    ))

events_by_minute_of_hour_dict

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(
    events_by_minute_of_hour["minute_of_hour"],
    events_by_minute_of_hour["events"],
    marker="o"
)

plt.xticks(range(0, 60, 5))
plt.xlabel("Minute within the Hour")
plt.ylabel("Number of Events")
plt.title("Event Distribution by Minute within the Hour")

plt.grid(alpha=0.3)

plt.tight_layout()

plt.savefig(output_path / "events_per_minute_of_hour.png")

plt.show()

In [ ]:
largest_deltatimes_df = timestamps_df.nlargest(
    20,
    "deltatime"
)[["timestamp", "deltatime"]]

largest_deltatimes_df

In [ ]:
largest_deltatimes_df.iloc[0, :]

i = int(largest_deltatimes_df.iloc[0, :].name) # type: ignore

burst_example_df = timestamps_df.loc[i-5:i+5, ["timestamp", "deltatime"]]

burst_example_df.to_csv(output_path / "burst_example.csv")

burst_example_df